<a href="https://colab.research.google.com/github/yeye080/customs-hs-classifier/blob/main/customs_hs_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CLASIFICADOR INTELIGENTE DE PARTIDAS ARANCELARIAS (ENTORNO DE PRUEBAS)
# =====================================================================

import pandas as pd
from difflib import SequenceMatcher


# 1. BASE DE DATOS SINTÉTICA (Protección Datos y Privacidad)

# Uso de datos genéricos y simulados para evitar el uso de información confidencial.
datos_sinteticos = {
    'HS_Code': ['8471.30.00', '6403.91.11', '0901.21.00', '2204.21.11', '8517.13.00', '9503.00.49', '3926.90.97', '9018.90.84'],
    'Categoria_Generica': ['Equipos Informáticos', 'Calzado Industrial/Seguridad', 'Materia Prima Agrícola (Café)', 'Bebidas Alcohólicas Embotelladas', 'Dispositivos de Comunicación', 'Artículos de Entretenimiento (Plástico)', 'Manufacturas de Plástico Genéricas', 'Instrumental Médico de Diagnóstico'],
    'Keywords': ['ordenador laptop notebook pc computadora pantalla', 'zapatos botas calzado cuero montaña proteccion', 'cafe grano molido tostado cafeina', 'vino alcohol botella tinto blanco bodega', 'movil telefono smartphone celular wifi', 'juguete plastico figura muñeco infantil', 'caja plastico envase tubo pieza pvc', 'medico hospital pinza bisturi jeringa estetoscopio'],
    'Arancel_Estimado_%': [0.0, 8.0, 7.5, 12.0, 0.0, 4.7, 6.5, 0.0],
    'IVA_Estimado_%': [21.0, 21.0, 10.0, 21.0, 21.0, 21.0, 21.0, 4.0],
    'Restricciones_Simuladas': ['Certificado de Seguridad Electrónica', 'Inspección de Calidad de Materiales', 'Certificado Fitosanitario de Importación', 'Licencia de Importación de Alcoholes', 'Homologación de Frecuencias', 'Certificación Infantil de No Toxicidad', 'Declaración de Impacto Ambiental (Plásticos)', 'Licencia de Importación Sanitaria / Marcado Médico']
}

df_aduanas_mock = pd.DataFrame(datos_sinteticos)



# 2. MOTOR DE BÚSQUEDA APROXIMADA (Fuzzy Matching)

def calcular_similitud(texto1, texto2):
    """Calcula el ratio de parecido entre dos cadenas de texto."""
    return SequenceMatcher(None, texto1.lower(), texto2.lower()).ratio()

def clasificador_aduanero_inteligente(busqueda_usuario, valor_mercancia=0.0, coste_envio=0.0):
    palabra_buscar = busqueda_usuario.lower().strip()
    mejor_coincidencia = None
    mejor_puntuacion = 0.0

    # Buscar la mejor coincidencia comparando con categorías y palabras clave
    for _, fila in df_aduanas_mock.iterrows():
        score_categoria = calcular_similitud(palabra_buscar, fila['Categoria_Generica'])

        score_keywords = 0.0
        for kw in fila['Keywords'].split():
            score_kw = calcular_similitud(palabra_buscar, kw)
            if score_kw > score_keywords:
                score_keywords = score_kw

        score_final = max(score_categoria, score_keywords)

        if score_final > mejor_puntuacion:
            mejor_puntuacion = score_final
            mejor_coincidencia = fila

    # Umbral de confianza (mínimo 60% de coincidencia)
    if mejor_coincidencia is None or mejor_puntuacion < 0.6:
        print(f"❌ No se pudo clasificar '{busqueda_usuario}' con suficiente seguridad.")
        print("💡 Prueba con términos como: 'computadora', 'bota', 'café', 'celular', 'bisturí' o 'botella'.")
        return


    # 3. LIQUIDACIÓN DE IMPUESTOS EN ADUANAS (CIF)

    valor_cif = valor_mercancia + coste_envio
    arancel_pagar = valor_cif * (mejor_coincidencia['Arancel_Estimado_%'] / 100.0)
    base_iva = valor_cif + arancel_pagar
    iva_pagar = base_iva * (mejor_coincidencia['IVA_Estimado_%'] / 100.0)
    total_impuestos = arancel_pagar + iva_pagar


    # 4. SALIDA DE DATOS FORMATEADA

    print("=" * 70)
    print("🛡️  CLASIFICACIÓN ARANCELARIA AUTOMÁTICA")
    print("=" * 70)
    print(f"🔍 Búsqueda:                 '{busqueda_usuario}'")
    print(f"🎯 Categoría Identificada:   {mejor_coincidencia['Categoria_Generica']} ({mejor_puntuacion*100:.1f}% confianza)")
    print(f"🔢 Código HS Simulado:       {mejor_coincidencia['HS_Code']}")
    print(f"📋 Requisitos de Aduana:     {mejor_coincidencia['Restricciones_Simuladas']}")
    print("-" * 70)

    if valor_mercancia > 0:
        print("💰 SIMULACIÓN DE LIQUIDACIÓN TRIBUTARIA (CIF):")
        print(f"• Valor de Mercancía:        {valor_mercancia:.2f} €")
        print(f"• Logística (Flete/Seguro):   {coste_envio:.2f} €")
        print(f"• Arancel Estimado ({mejor_coincidencia['Arancel_Estimado_%']}%):  {arancel_pagar:.2f} €")
        print(f"• IVA Estimado ({mejor_coincidencia['IVA_Estimado_%']}%):      {iva_pagar:.2f} €")
        print(f"🚨 TOTAL LIQUIDACIÓN ADUANA: {total_impuestos:.2f} €")
        print("=" * 70)


# 5. PRUEBAS DE FUNCIONAMIENTO EN VIVO

# Prueba 1: Buscando con error de escritura ("compuadora")
clasificador_aduanero_inteligente("compuadora", valor_mercancia=3500.0, coste_envio=250.0)

print("\n")

# Prueba 2: Buscando por un sinónimo que no está en el título principal ("botas")
clasificador_aduanero_inteligente("botas", valor_mercancia=12000.0, coste_envio=1100.0)

🛡️  CLASIFICACIÓN ARANCELARIA AUTOMÁTICA
🔍 Búsqueda:                 'compuadora'
🎯 Categoría Identificada:   Equipos Informáticos (95.2% confianza)
🔢 Código HS Simulado:       8471.30.00
📋 Requisitos de Aduana:     Certificado de Seguridad Electrónica
----------------------------------------------------------------------
💰 SIMULACIÓN DE LIQUIDACIÓN TRIBUTARIA (CIF):
• Valor de Mercancía:        3500.00 €
• Logística (Flete/Seguro):   250.00 €
• Arancel Estimado (0.0%):  0.00 €
• IVA Estimado (21.0%):      787.50 €
🚨 TOTAL LIQUIDACIÓN ADUANA: 787.50 €


🛡️  CLASIFICACIÓN ARANCELARIA AUTOMÁTICA
🔍 Búsqueda:                 'botas'
🎯 Categoría Identificada:   Calzado Industrial/Seguridad (100.0% confianza)
🔢 Código HS Simulado:       6403.91.11
📋 Requisitos de Aduana:     Inspección de Calidad de Materiales
----------------------------------------------------------------------
💰 SIMULACIÓN DE LIQUIDACIÓN TRIBUTARIA (CIF):
• Valor de Mercancía:        12000.00 €
• Logística (Flete/Seguro):